# FACTOR Demo：多智能体驱动的空间组学双侧受控因子元基准框架

> **汇报核心主线**：传统空间组学 Benchmark 只能看静态排行榜或做单侧消融，无法区分“模型架构真正适配数据特征”还是“预处理/默认超参混杂”。本 Demo 实现了三组协同智能体（**Dataset-Character Agent**、**Method-Character Agent**、**D–M Interaction & Counterfactual Agent**），打通从**跨切片特征量化 $\rightarrow$ AST 源码级归纳偏置解析 $\rightarrow$ 观测交互筛选 $\rightarrow$ 独立样本 $2\times 3$ 双侧受控反事实实验 $\rightarrow$ 自动导出 Markdown/PDF 实验报告**的完整闭环。

### 核心设计亮点（内置严谨对照护栏）
1. **确定性计算与智能体编排分离**：所有空间统计量、混合效应回归、置信区间与等效检验均由确定性 Python 工具执行，杜绝大模型数值幻觉。
2. **AST 源码行号溯源 + 零干预代码等价门控**：`MethodCharacterAgent` 直接解析算法 `.py` 语法树绑定行号，并在干预前强制校验 `orig == refac`。
3. **四臂方法对照与双调参协议**：分离 `refac`（真实空间图）、`self`（单位阵自环 $\mathbf{I}$）与 `null`（同谱节点置换图 $\mathbf{P}\mathbf{A}_{\text{spat}}\mathbf{P}^\top$），同步对比默认配置（`fixed`）与等预算重调优（`retuned`），并直接做 `PCA vs. HVG` 预处理交叉验证。

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from factor_demo.llm_client import DualModeLLMClient
from factor_demo.data_lab.generator_and_loader import build_demo_cohort
from factor_demo.methods_zoo.registry import METHODS_REGISTRY
from factor_demo.agents import DatasetCharacterAgent, MethodCharacterAgent, DMInteractionAgent

# 双模式配置：若环境变量设置了 OPENAI_API_KEY（支持 DeepSeek / GPT / Gemini 兼容端点），自动走实时 API；否则走零翻车离线工具溯源模式
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com/v1"
# os.environ["LLM_MODEL"] = "deepseek-chat"

llm = DualModeLLMClient(mode="auto")
print("当前智能体引擎状态:", "Live API (" + llm.model + ")" if llm.is_live_available else "Offline Tool-Grounded Mode")

## Step 1：`Dataset-Character Agent` 自动提取切片元特征（严格隔离无标签特征 `d_obs` 与诊断特征 `d_ref`）

In [ ]:
cohort = build_demo_cohort(seed=42)
expl_slices = [s for s in cohort if s.split == "exploration"]
conf_slices = [s for s in cohort if s.split == "confirmation"]

dataset_agent = DatasetCharacterAgent(llm_client=llm)
d_profiles = dataset_agent.profile_cohort(cohort)

df_d = pd.DataFrame([
    {
        "slice_id": p.dataset_id,
        "split": p.split,
        "platform": p.platform_tag,
        "tissue": p.tissue_tag,
        **{f"obs:{k}": v for k, v in p.d_obs.items()},
        **{f"ref:{k}": v for k, v in p.d_ref.items()},
    }
    for p in d_profiles
])
display(df_d)
print("\n[Agent 1 切片诊断卡片示例]")
for p in d_profiles[:3]:
    print(" -", p.agent_summary)

## Step 2：`Method-Character Agent` 静态解析算法源码 AST 并执行零干预代码等价校验

In [ ]:
method_ids = list(METHODS_REGISTRY.keys())
method_agent = MethodCharacterAgent(llm_client=llm)
m_profiles = method_agent.profile_methods(method_ids)

df_m = pd.DataFrame([
    {
        "method_id": mp.method_id,
        "family": mp.family,
        **mp.m_core,
        **mp.c_aux,
        "code_citations": "; ".join(f"{v.file_path}:L{v.start_line}-{v.end_line}" for v in mp.evidence_spans.values()),
    }
    for mp in m_profiles
])
display(df_m)

eq_check = method_agent.verify_code_equivalence(conf_slices[0], method_ids, seed=0)
print("代码重构等价对照门控 (orig vs. refac 最大差异, 0.0 表示严格一致):", eq_check)
for mp in m_profiles:
    print(" -", mp.agent_summary)

## Step 3：`D–M Interaction Agent` 外循环筛选与预注册假设锁定（仅使用探索集 `D_expl`）

In [ ]:
interaction_agent = DMInteractionAgent(
    llm_client=llm,
    dataset_agent=dataset_agent,
    method_agent=method_agent,
    n_pert_seeds=2,
    n_null_seeds=2,
    n_train_seeds=3,
    equivalence_margin=0.03,
)

expl_profiles = [p for p in d_profiles if p.split == "exploration"]
conf_profiles = [p for p in d_profiles if p.split == "confirmation"]

y_expl, _ = interaction_agent.run_observational_benchmark(expl_slices, method_ids)
candidates, hypotheses = interaction_agent.propose_and_preregister_hypotheses(
    expl_profiles=expl_profiles,
    expl_slices=expl_slices,
    method_profiles=m_profiles,
    y_expl=y_expl,
)

display(pd.DataFrame([c.__dict__ for c in candidates[:8]]))
for h in hypotheses:
    print(f"\n[预注册锁定假设 {h.hypothesis_id}] {h.title}")
    print("  依据:", h.rationale_from_screening)
    print("  证伪判据:", h.falsification_rule)

## Step 4：内循环双侧 $2\times 3$ 受控因子实验、偏移审计与样本外推荐评测（独立确认集 `D_conf`）

In [ ]:
from factor_demo.pipeline import run_demo_pipeline

# 一键执行完整管线并导出图表、Markdown 报告与多页矢量 PDF 报告
results = run_demo_pipeline(
    output_dir=PROJECT_ROOT / "outputs" / "notebook_demo_run",
    llm_mode=llm.mode,
    n_pert_seeds=1,
    n_null_seeds=1,
    n_train_seeds=2,
    verbose=True,
)

df_contrasts = pd.DataFrame(results["bundle"]["confirmatory_contrasts"])
display(df_contrasts[[
    "hypothesis_id", "method_id", "tuning_protocol",
    "simple_topo_low", "simple_topo_high", "tau_int_topo_mean",
    "tau_int_topo_ci_low", "tau_int_topo_ci_high", "evidence_state"
]])

## Step 5：内联展示生成的 4 张核心验证图表与报告路径

In [ ]:
for key in ("fig1", "fig2", "fig3", "fig4"):
    display(Image(filename=str(results["figures"][key]), width=920))

print("生成的 Markdown 完整实验报告:", results["markdown_report"])
print("生成的多页矢量 PDF 实验报告:", results["pdf_report"])